# InternVL3.5 Kaggle smoke test

This notebook tests the official post-trained `OpenGVLab/InternVL3_5-8B-HF` checkpoint on **three existing pilot2 task-condition pairs**: T-dir, R-dir, and G-dir. It does not run a full pilot or main study. Enable a Kaggle GPU and Internet, then attach the private completed Qwen archive (`visdsr_results.zip`) as the only VisDSR input. Run All, review the three responses, and download the unique export from `/kaggle/working`. The existing tasks, images, prompts, DSU rules, and strict scorer are reused without edits.


In [ ]:
REPOSITORY = "https://github.com/Abhishekgit01/VisDSR.git"


## Restore the project and completed calibration data


In [ ]:
from pathlib import Path
import shutil
import socket
import subprocess
import sys
import zipfile

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
ROOT = WORK / "VisDSR"

def run_command(*parts, cwd=None):
    subprocess.run(parts, cwd=cwd or ROOT, check=True)

def safe_extract(archive, destination):
    with zipfile.ZipFile(archive) as source:
        selected = [member for member in source.infolist()
                    if member.filename.startswith(("data/", "results/"))]
        for member in selected:
            path = (destination / member.filename).resolve()
            if not path.is_relative_to(destination.resolve()):
                raise ValueError(f"unsafe archive path: {member.filename}")
        source.extractall(destination, members=selected)

if not (ROOT / "configs/experiment.yaml").exists():
    if not REPOSITORY.startswith("https://github.com/") or any(c in REPOSITORY for c in "[]()"):
        raise ValueError("REPOSITORY must be a plain GitHub URL")
    try:
        socket.getaddrinfo("github.com", 443)
    except OSError as exc:
        raise RuntimeError("Enable Internet in Kaggle Session options, restart, then Run All") from exc
    run_command("git", "clone", "--depth", "1", REPOSITORY, str(ROOT), cwd=WORK)
print("Project root:", ROOT)

INPUT = Path("/kaggle/input")
if not INPUT.exists():
    raise RuntimeError("Attach the completed private VisDSR results Dataset in Kaggle Input")
archives = []
for path in INPUT.rglob("*.zip"):
    try:
        with zipfile.ZipFile(path) as source:
            if {"results/scores_pilot2_model1.csv", "data/pilot2/tasks.jsonl"}.issubset(source.namelist()):
                archives.append(path)
    except zipfile.BadZipFile:
        continue
extracted = [p.parent.parent for p in INPUT.rglob("scores_pilot2_model1.csv")
             if p.parent.name == "results" and
             (p.parent.parent / "data/pilot2/tasks.jsonl").exists()]
if archives:
    if len(archives) != 1:
        raise RuntimeError("Attach only the latest VisDSR results archive")
    safe_extract(archives[0], ROOT)
    print("Restored archive:", archives[0])
elif extracted:
    if len(extracted) != 1:
        raise RuntimeError("Attach only one extracted VisDSR results Dataset")
    for folder in ("data", "results"):
        shutil.copytree(extracted[0] / folder, ROOT / folder, dirs_exist_ok=True)
    print("Restored extracted results:", extracted[0])
else:
    raise RuntimeError("Attach visdsr_results.zip as a private Kaggle Dataset")
for name in ("data/pilot2/tasks.jsonl", "data/pilot2/manifest.csv",
             "results/scores_pilot2_model1.csv"):
    if not (ROOT / name).exists():
        raise RuntimeError(f"Completed Qwen calibration file missing: {name}")


## Install dependencies and check the GPU


In [ ]:
import platform
import os
print("Python:", platform.python_version())
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before Run All")
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"GPU {index}: {props.name}; VRAM {props.total_memory / 2**30:.2f} GiB")
run_command(sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "environment/requirements.txt"))
run_command(sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed",
            "transformers==4.57.1", "accelerate", "bitsandbytes", "huggingface_hub")
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
import transformers
import yaml
settings = next((m for m in yaml.safe_load((ROOT / "configs/experiment.yaml").read_text())["models"]
                 if m["name"] == "model2"), None)
if settings is None or settings["id"] != "OpenGVLab/InternVL3_5-8B-HF":
    raise RuntimeError("The attached project copy is outdated; import the latest GitHub notebook and repository")
print("Transformers:", transformers.__version__)
print("Model:", settings["id"])
print("Pinned revision:", settings["revision"])
print("Generation settings:", {k: settings[k] for k in
                               ("do_sample", "num_beams", "max_output_tokens", "quantization")})


## Validate the unchanged pilot2 tasks and images


In [ ]:
fonts = [Path("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"),
         Path("/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf")]
if shutil.which("dot") is None or not all(path.exists() for path in fonts):
    run_command("apt-get", "update", "-qq")
    run_command("apt-get", "install", "-y", "-qq", "graphviz", "fonts-dejavu-core")
run_command("make", "build")
run_command(sys.executable, "-m", "gen.validate", "--split", "pilot2")
print("Unchanged pilot2 tasks and images ready")


## Three-call smoke test

This cell makes one text call and two image calls. The runner immediately saves each raw response and refreshes its cache snapshot. Review its strict parse results, GPU use, latency, and exact revision before any larger run.


In [ ]:
run_command(sys.executable, "-m", "eval.run", "--split", "pilot2",
            "--model", "model2", "--smoke")
import csv
with (ROOT / "results/scores_pilot2_smoke_model2.csv").open(newline="") as source:
    smoke_rows = list(csv.DictReader(source))
if len(smoke_rows) != 3 or {row["condition"] for row in smoke_rows} != {"T-dir", "R-dir", "G-dir"}:
    raise RuntimeError("The InternVL smoke results are incomplete")
print("STOP: review these three InternVL responses before any full pilot.")


## Export and download

Run this cell after the smoke test, or after a partial failure to preserve cached responses. Download `visdsr_internvl35_smoke.zip` from Kaggle Output before leaving the session. Do not download the older ZIP from Input.


In [ ]:
run_command(sys.executable, "-m", "manifest")
export = WORK / "visdsr_internvl35_smoke.zip"
with zipfile.ZipFile(export, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for folder in ("results", "data"):
        for path in sorted((ROOT / folder).rglob("*")):
            if path.is_file() and path.relative_to(ROOT).as_posix() != "results/cache_snapshot.zip":
                archive.write(path, path.relative_to(ROOT))
    for name in ("MANIFEST.json", "configs/experiment.yaml", "CHANGELOG.md"):
        path = ROOT / name
        if path.exists():
            archive.write(path, name)
with zipfile.ZipFile(export) as check:
    print("InternVL smoke scores in export:",
          "results/scores_pilot2_smoke_model2.csv" in check.namelist())
print("Export:", export)
